# 🚀 Spaceship Titanic — Ensemble Learning Homework

**Bài toán:** Dự đoán hành khách có bị "Transported" (dịch chuyển sang chiều không gian khác) hay không, dựa trên dữ liệu của cuộc thi Kaggle [Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic).

**Nội dung notebook:**
1. Tải & khám phá dữ liệu (EDA)
2. Tiền xử lý & Feature Engineering
3. Xây dựng các mô hình baseline (không dùng Ensemble)
4. Ensemble Learning:
   - **Bagging**: `BaggingClassifier`, `RandomForest`, `ExtraTrees`
   - **Boosting**: `AdaBoost`, `GradientBoosting`, `HistGradientBoosting` (và tùy chọn `XGBoost`/`LightGBM` nếu máy bạn có cài)
   - **Stacking**: `StackingClassifier`
5. So sánh benchmark giữa các mô hình
6. Tinh chỉnh siêu tham số (Hyperparameter Tuning) cho mô hình tốt nhất
7. Huấn luyện lại trên toàn bộ tập train và dự đoán trên tập test → xuất file `submission.csv` nộp lên Kaggle

> ⚠️ **Lưu ý quan trọng trước khi chạy:** Notebook này cần 2 file `train.csv` và `test.csv` của cuộc thi. Xem hướng dẫn tải dữ liệu ở cell tiếp theo.


## 📥 Bước 0: Tải dữ liệu

1. Vào trang cuộc thi: https://www.kaggle.com/competitions/spaceship-titanic/data
2. Bấm **Download All** để tải file `spaceship-titanic.zip` (cần đăng nhập Kaggle và bấm "Join Competition" nếu chưa tham gia).
3. Giải nén, bạn sẽ có 3 file: `train.csv`, `test.csv`, `sample_submission.csv`.
4. Copy 3 file này vào **cùng thư mục** với file notebook (`.ipynb`) này.

Nếu bạn dùng Kaggle Notebook thay vì Jupyter local, chỉ cần "Add Data" → chọn cuộc thi `spaceship-titanic`, đường dẫn sẽ là `/kaggle/input/spaceship-titanic/train.csv`. Bạn có thể sửa lại `DATA_DIR` ở cell dưới cho phù hợp.


In [ ]:
# Nếu chạy trên Kaggle Notebook, đổi DATA_DIR thành "/kaggle/input/spaceship-titanic/"
DATA_DIR = "./"


## 📦 Bước 1: Import thư viện

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("future.infer_string", False)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, RandomizedSearchCV
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier,
    AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier,
    StackingClassifier, VotingClassifier
)

HAS_XGB = False
HAS_LGBM = False
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    print("Chưa cài xgboost -> bỏ qua (không bắt buộc). Cài bằng: pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    print("Chưa cài lightgbm -> bỏ qua (không bắt buộc). Cài bằng: pip install lightgbm")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 📂 Bước 2: Đọc dữ liệu

In [ ]:
train = pd.read_csv(DATA_DIR + "train.csv")
test = pd.read_csv(DATA_DIR + "test.csv")

print("Kích thước tập train:", train.shape)
print("Kích thước tập test :", test.shape)
train.head()


## 🔎 Bước 3: Khám phá dữ liệu (EDA)

**Mô tả các cột trong bộ dữ liệu:**
- `PassengerId`: Mã hành khách, dạng `gggg_pp` (gggg = mã nhóm/gia đình, pp = số thứ tự trong nhóm).
- `HomePlanet`: Hành tinh xuất phát.
- `CryoSleep`: Hành khách có ở trạng thái ngủ đông trong suốt chuyến đi hay không.
- `Cabin`: Số phòng dạng `deck/num/side` (side = P: Port hoặc S: Starboard).
- `Destination`: Hành tinh đến.
- `Age`: Tuổi.
- `VIP`: Có mua dịch vụ VIP hay không.
- `RoomService, FoodCourt, ShoppingMall, Spa, VRDeck`: Số tiền đã chi cho từng dịch vụ tiện ích.
- `Name`: Họ tên.
- `Transported` (**biến mục tiêu**): Hành khách có bị dịch chuyển sang chiều không gian khác hay không.


In [ ]:
train.info()


In [ ]:
missing = train.isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]
plt.figure(figsize=(8, 5))
sns.barplot(x=missing.values, y=missing.index, color="steelblue")
plt.xlabel("% giá trị bị thiếu")
plt.title("Tỉ lệ missing value theo cột")
plt.show()


In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="Transported", data=train)
plt.title("Phân phối biến mục tiêu Transported")
plt.show()

train["Transported"].value_counts(normalize=True)


In [ ]:
num_features = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), num_features):
    sns.histplot(train[col].dropna(), bins=30, ax=ax, kde=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()


In [ ]:
cat_features = ["HomePlanet", "CryoSleep", "Destination", "VIP"]
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.flatten(), cat_features):
    sns.countplot(x=col, hue="Transported", data=train, ax=ax)
    ax.set_title(f"{col} vs Transported")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
corr = train[num_features].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Ma trận tương quan các đặc trưng số")
plt.show()


**Một vài quan sát ban đầu (dựa trên đặc điểm đã biết của bộ dữ liệu này):**
- Hành khách ở trạng thái `CryoSleep = True` gần như không chi tiêu gì cho các dịch vụ tiện ích (`RoomService`, `Spa`,...) → có thể dùng để suy luận/điền giá trị thiếu cho `CryoSleep`.
- Các cột chi tiêu (`RoomService`, `FoodCourt`,...) có phân phối lệch phải rất mạnh (nhiều giá trị 0).
- `Cabin` chứa 3 thông tin gộp lại (deck/num/side) nên cần tách ra để mô hình khai thác tốt hơn.
- `PassengerId` chứa thông tin nhóm hành khách (đi cùng gia đình/nhóm) — số lượng người trong 1 nhóm có thể là đặc trưng hữu ích.


## 🛠️ Bước 4: Feature Engineering & Tiền xử lý

Các đặc trưng mới được tạo thêm:
- **`Deck`, `CabinNum`, `Side`**: tách từ `Cabin`.
- **`Group`, `GroupSize`, `IsAlone`**: tách từ `PassengerId` — số người trong cùng 1 nhóm (được tính trên **cả train + test** vì đây chỉ là thông tin về "nhóm đi cùng nhau", không dùng đến nhãn `Transported` nên không gây rò rỉ dữ liệu (data leakage)).
- **`TotalSpend`, `HasSpent`**: tổng chi tiêu và cờ đánh dấu có chi tiêu hay không.
- Điền khuyết `CryoSleep`: nếu thiếu và tổng chi tiêu = 0 → khả năng cao là `True`.
- Điền khuyết `VIP`: mặc định `False` (VIP là thiểu số).


In [ ]:
all_ids = pd.concat([train["PassengerId"], test["PassengerId"]])
all_groups = all_ids.str.split("_", expand=True)[0]
group_size_map = all_groups.value_counts().to_dict()


def feature_engineer(df):
    df = df.copy()

    cabin_split = df["Cabin"].str.split("/", expand=True)
    df["Deck"] = cabin_split[0]
    df["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
    df["Side"] = cabin_split[2]

    df["Group"] = df["PassengerId"].str.split("_", expand=True)[0]
    df["GroupSize"] = df["Group"].map(group_size_map)
    df["IsAlone"] = (df["GroupSize"] == 1).astype(int)

    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    for c in spend_cols:
        df[c] = df[c].fillna(0)
    df["TotalSpend"] = df[spend_cols].sum(axis=1)
    df["HasSpent"] = (df["TotalSpend"] > 0).astype(int)

    df["CryoSleep"] = df["CryoSleep"].astype("object")
    mask = df["CryoSleep"].isna() & (df["TotalSpend"] == 0)
    df.loc[mask, "CryoSleep"] = True
    df["CryoSleep"] = df["CryoSleep"].fillna(False).astype(bool)

    df["VIP"] = df["VIP"].fillna(False).astype(bool)

    df = df.drop(columns=["PassengerId", "Cabin", "Name", "Group"])
    return df


train_fe = feature_engineer(train)
test_ids = test["PassengerId"].copy()
test_fe = feature_engineer(test)

train_fe.head()


In [ ]:
X = train_fe.drop(columns=["Transported"])
y = train_fe["Transported"].astype(int)
X_test_final = test_fe

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()
print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)


Xây dựng **pipeline tiền xử lý** dùng `ColumnTransformer`:
- Cột số: điền khuyết bằng median + chuẩn hoá `StandardScaler`.
- Cột phân loại: điền khuyết bằng mode (giá trị xuất hiện nhiều nhất) + `OneHotEncoder`.

Pipeline này sẽ được gắn liền với từng mô hình để tránh rò rỉ dữ liệu (data leakage) giữa tập train/validation.


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])


## ✂️ Bước 5: Chia tập train/validation

Chia 80/20, giữ nguyên tỉ lệ nhãn (`stratify=y`) để đánh giá các mô hình một cách công bằng trước khi nộp bài lên Kaggle.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_val.shape)


## 🧪 Bước 6: Các mô hình Baseline (chưa dùng Ensemble)

So sánh với 3 mô hình đơn giản, không phải kỹ thuật ensemble, để làm mốc so sánh (baseline):
- Logistic Regression
- Decision Tree
- K-Nearest Neighbors


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}


def evaluate_model(name, model):
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
    val_acc = accuracy_score(y_val, val_pred)
    results[name] = {
        "CV Accuracy (mean)": cv_scores.mean(),
        "CV Accuracy (std)": cv_scores.std(),
        "Validation Accuracy": val_acc
    }
    print(f"{name:22s} | CV acc: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f}) | Val acc: {val_acc:.4f}")
    return pipe


baseline_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

fitted_pipes = {}
for name, model in baseline_models.items():
    fitted_pipes[name] = evaluate_model(name, model)


## 🎒 Bước 7: Ensemble Learning — Bagging

**Bagging (Bootstrap Aggregating)**: huấn luyện nhiều mô hình con trên các tập con dữ liệu được lấy mẫu ngẫu nhiên có hoàn lại (bootstrap), sau đó lấy kết quả trung bình/biểu quyết để giảm phương sai (variance) và tránh overfitting.

Các mô hình sử dụng:
- `BaggingClassifier` với base estimator là `DecisionTreeClassifier`
- `RandomForestClassifier` (bản chất là Bagging trên Decision Tree + chọn ngẫu nhiên đặc trưng)
- `ExtraTreesClassifier` (giống Random Forest nhưng chọn ngưỡng chia ngẫu nhiên hơn)


In [ ]:
bagging_models = {
    "Bagging (Decision Tree)": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

for name, model in bagging_models.items():
    fitted_pipes[name] = evaluate_model(name, model)


## 🚀 Bước 8: Ensemble Learning — Boosting

**Boosting**: huấn luyện tuần tự nhiều mô hình yếu (weak learner), mô hình sau tập trung sửa lỗi của mô hình trước, giúp giảm bias.

Các mô hình sử dụng:
- `AdaBoostClassifier`
- `GradientBoostingClassifier`
- `HistGradientBoostingClassifier` (bản tối ưu tốc độ của Gradient Boosting, lấy cảm hứng từ LightGBM)
- (Bonus, nếu có cài) `XGBoost`, `LightGBM`


In [ ]:
boosting_models = {
    "AdaBoost": AdaBoostClassifier(n_estimators=300, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=300, max_depth=3, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=300, random_state=RANDOM_STATE),
}

if HAS_XGB:
    boosting_models["XGBoost"] = XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
    )

if HAS_LGBM:
    boosting_models["LightGBM"] = LGBMClassifier(
        n_estimators=400, max_depth=-1, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )

for name, model in boosting_models.items():
    fitted_pipes[name] = evaluate_model(name, model)


## 🧩 Bước 9: Ensemble Learning — Stacking

**Stacking**: huấn luyện nhiều mô hình cơ sở (base learners) khác loại, sau đó dùng một mô hình meta (meta-learner) để học cách kết hợp dự đoán của các mô hình cơ sở lại với nhau, thường cho kết quả tốt hơn từng mô hình đơn lẻ vì tận dụng được điểm mạnh của nhiều loại mô hình khác nhau.

Ở đây, ta kết hợp: Random Forest + Gradient Boosting + SVM, với meta-learner là Logistic Regression. Ta cũng so sánh thêm với `VotingClassifier` (biểu quyết mềm - soft voting, không có meta-learner) để thấy rõ sự khác biệt.


In [ ]:
base_estimators = [
    ("rf", RandomForestClassifier(n_estimators=300, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)),
    ("gb", GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=RANDOM_STATE)),
    ("svc", SVC(probability=True, random_state=RANDOM_STATE)),
]

stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5, n_jobs=-1
)
fitted_pipes["Stacking"] = evaluate_model("Stacking", stacking_model)

voting_model = VotingClassifier(estimators=base_estimators, voting="soft", n_jobs=-1)
fitted_pipes["Voting (soft)"] = evaluate_model("Voting (soft)", voting_model)


## 📊 Bước 10: Bảng so sánh Benchmark

So sánh toàn bộ các mô hình: baseline, Bagging, Boosting, Stacking/Voting theo Cross-Validation Accuracy và Validation Accuracy.


In [ ]:
results_df = pd.DataFrame(results).T.sort_values("CV Accuracy (mean)", ascending=False)
results_df


In [ ]:
plt.figure(figsize=(10, 6))
order = results_df.index
sns.barplot(x=results_df["CV Accuracy (mean)"], y=order, color="teal")
plt.xlabel("CV Accuracy (mean, 5-fold)")
plt.title("So sánh benchmark các mô hình")
plt.xlim(0.5, 1.0)
plt.show()


> 📝 **Nhận xét chung** (thường thấy trên bộ dữ liệu Spaceship Titanic thực tế):
> - Các mô hình Ensemble (Bagging/Boosting/Stacking) thường vượt trội hơn hẳn các mô hình baseline đơn lẻ (Logistic Regression, Decision Tree, KNN).
> - Boosting (Gradient Boosting/HistGradientBoosting/XGBoost/LightGBM) thường cho kết quả tốt nhất trên bộ dữ liệu dạng bảng (tabular) như thế này.
> - Stacking thường cho kết quả ổn định, tương đương hoặc nhỉnh hơn mô hình boosting tốt nhất, nhờ kết hợp được điểm mạnh của nhiều loại mô hình.
> - Kết quả cụ thể (thứ hạng mô hình nào cao nhất) có thể thay đổi tuỳ theo dữ liệu train/validation split và random_state — hãy chạy lại với dữ liệu thật và ghi nhận con số thực tế của bạn ở đây.


## 🔧 Bước 11: Tinh chỉnh siêu tham số (Hyperparameter Tuning)

Chọn ra mô hình có kết quả tốt nhất ở bảng benchmark phía trên, sau đó dùng `RandomizedSearchCV` để tìm bộ tham số tốt hơn (nhanh hơn `GridSearchCV` toàn diện nhưng vẫn hiệu quả).

👉 Mặc định bên dưới đang tinh chỉnh cho `HistGradientBoostingClassifier` — bạn có thể đổi sang mô hình khác (ví dụ Stacking, XGBoost, LightGBM,...) nếu mô hình đó cho kết quả benchmark tốt nhất với dữ liệu thật của bạn.


In [ ]:
param_distributions = {
    "model__max_iter": [200, 300, 400, 600],
    "model__max_depth": [None, 3, 5, 7, 10],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_leaf_nodes": [15, 31, 63, 127],
    "model__l2_regularization": [0.0, 0.1, 0.5, 1.0],
}

tuning_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
])

random_search = RandomizedSearchCV(
    tuning_pipe,
    param_distributions=param_distributions,
    n_iter=30,
    cv=cv,
    scoring="accuracy",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
random_search.fit(X_train, y_train)

print("Bộ tham số tốt nhất:", random_search.best_params_)
print("CV Accuracy tốt nhất:", random_search.best_score_)

best_pipe = random_search.best_estimator_
val_acc = accuracy_score(y_val, best_pipe.predict(X_val))
print("Validation Accuracy sau tinh chỉnh:", val_acc)


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, best_pipe.predict(X_val))
plt.title("Confusion Matrix - Best model (validation set)")
plt.show()

print(classification_report(y_val, best_pipe.predict(X_val), target_names=["Not Transported", "Transported"]))


## 🏁 Bước 12: Huấn luyện lại trên toàn bộ tập train & Dự đoán tập test

Sau khi đã chọn được mô hình tốt nhất (và bộ tham số tốt nhất), ta huấn luyện lại **trên toàn bộ tập train** (gộp cả phần validation) để tận dụng tối đa dữ liệu, sau đó dự đoán trên tập test thật của cuộc thi và xuất file `submission.csv` để nộp lên Kaggle.


In [ ]:
final_model = random_search.best_estimator_
final_model.fit(X, y)

test_predictions = final_model.predict(X_test_final)

submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": test_predictions.astype(bool)
})

submission.to_csv("submission.csv", index=False)
print("Đã lưu file submission.csv, kích thước:", submission.shape)
submission.head()


## 📝 Kết luận & Hướng cải thiện thêm

**Tóm tắt:**
- Đã xây dựng và so sánh 3 nhóm mô hình Ensemble: **Bagging** (BaggingClassifier, Random Forest, Extra Trees), **Boosting** (AdaBoost, Gradient Boosting, HistGradientBoosting, XGBoost/LightGBM nếu có) và **Stacking**.
- Bảng benchmark ở Bước 10 cho thấy các mô hình Ensemble vượt trội so với các mô hình baseline đơn lẻ.
- Mô hình tốt nhất đã được tinh chỉnh siêu tham số bằng `RandomizedSearchCV` và dùng để dự đoán tập test, xuất ra `submission.csv`.

**Cách nộp bài lên Kaggle:**
1. Vào https://www.kaggle.com/competitions/spaceship-titanic/submit
2. Upload file `submission.csv` vừa tạo.
3. Ghi lại điểm số (score) hiển thị trên leaderboard và điền vào báo cáo.

**Một vài hướng để cải thiện điểm số hơn nữa (nếu điểm hiện tại chưa đạt >80%):**
- Thử thêm đặc trưng: nhóm tuổi (`Age` binning), số người *cùng họ* (surname) đi cùng nhau, tương tác giữa `Deck` và `HomePlanet`,...
- Thử `Target Encoding` cho các biến phân loại có nhiều nhóm (`Deck`, `HomePlanet`).
- Tăng `n_iter` của `RandomizedSearchCV` hoặc dùng `GridSearchCV` với vùng tham số hẹp hơn quanh giá trị tốt nhất tìm được.
- Cài thêm `xgboost`/`lightgbm`/`catboost` (`pip install xgboost lightgbm catboost`) — thường cho kết quả tốt hơn `HistGradientBoosting` trên bộ dữ liệu dạng bảng.
- Blend (trung bình dự đoán xác suất) của 2-3 mô hình boosting mạnh nhất thay vì chỉ dùng 1 mô hình.
- Thử `StackingClassifier` với nhiều base model boosting hơn (XGBoost + LightGBM + CatBoost) làm base learners.
